In [0]:
# OLIST - CAMADA SILVER
# Descrição: Leitura das tabelas brutas da camada Bronze, conversão de tipos de dados,
#            tratameto de nulos, padronização de textos e enriquecimento com de-paras.

from pyspark.sql.functions import col, trim, lower, upper, coalesce, lit, current_timestamp

# 1. CRIAÇÃO DO DATABASE/SCHEMA SILVER
# Garante que o schema de destino esteja criado no Unity Catalog
spark.sql("CREATE DATABASE IF NOT EXISTS silver_olist")
print("Database 'silver_olist' pronto para receber as tabelas.")

# 2. TABELA: SILVER_ORDERS (Pedidos)
# Objetivo: Converter colunas de string para TIMESTAMP para permitir cálculos de frete/atraso.
print("Processando: silver_orders...")

df_orders = spark.table("bronze_olist.bronze_orders")

df_orders_silver = df_orders.select(
    col("order_id").alias("order_id"),
    col("customer_id").alias("customer_id"),
    col("order_status").alias("order_status"),
    
    # Conversão de strings de data para tipo TIMESTAMP oficial
    col("order_purchase_timestamp").cast("timestamp").alias("order_purchase_timestamp"),
    col("order_approved_at").cast("timestamp").alias("order_approved_at"),
    col("order_delivered_carrier_date").cast("timestamp").alias("order_delivered_carrier_date"),
    col("order_delivered_customer_date").cast("timestamp").alias("order_delivered_customer_date"),
    col("order_estimated_delivery_date").cast("timestamp").alias("order_estimated_delivery_date"),
    
    # Metadado de processamento da camada Silver
    current_timestamp().alias("_processed_at")
)

# Salva/Sobrescreve a tabela Delta no schema Silver
df_orders_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_olist.silver_orders")


# 3. TABELA: SILVER_ORDER_ITEMS (Itens do Pedido)
# Objetivo: Garantir tipagem correta de valores monetários (double) e sequenciais (int).
print("Processando: silver_order_items...")

df_items = spark.table("bronze_olist.bronze_order_items")

df_items_silver = df_items.select(
    col("order_id").alias("order_id"),
    col("order_item_id").cast("integer").alias("order_item_id"),
    col("product_id").alias("product_id"),
    col("seller_id").alias("seller_id"),
    col("shipping_limit_date").cast("timestamp").alias("shipping_limit_date"),
    
    # Valores financeiros formatados para ponto flutuante de precisão
    col("price").cast("double").alias("price"),
    col("freight_value").cast("double").alias("freight_value"),
    
    current_timestamp().alias("_processed_at")
)

df_items_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_olist.silver_order_items")


# 4. TABELA: SILVER_CUSTOMERS (Clientes)
# Objetivo: Padronizar textos de localização (espaços e caixa alta/baixa).
print("Processando: silver_customers...")

df_customers = spark.table("bronze_olist.bronze_customers")

df_customers_silver = df_customers.select(
    col("customer_id").alias("customer_id"),
    col("customer_unique_id").alias("customer_unique_id"),
    col("customer_zip_code_prefix").cast("string").alias("customer_zip_code_prefix"),
    
    # Padronização de strings: remoção de espaços e conversão para minúsculas/maiúsculas
    trim(lower(col("customer_city"))).alias("customer_city"),
    trim(upper(col("customer_state"))).alias("customer_state"),
    
    current_timestamp().alias("_processed_at")
)

df_customers_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_olist.silver_customers")


# 5. TABELA: SILVER_SELLERS (Vendedores)
# Objetivo: Padronizar nomes de cidades e estados dos vendedores.
print("Processando: silver_sellers...")

df_sellers = spark.table("bronze_olist.bronze_sellers")

df_sellers_silver = df_sellers.select(
    col("seller_id").alias("seller_id"),
    col("seller_zip_code_prefix").cast("string").alias("seller_zip_code_prefix"),
    
    trim(lower(col("seller_city"))).alias("seller_city"),
    trim(upper(col("seller_state"))).alias("seller_state"),
    
    current_timestamp().alias("_processed_at")
)

df_sellers_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_olist.silver_sellers")


# 6. TABELA: SILVER_ORDER_PAYMENTS (Pagamentos)
# Objetivo: Tipar parcelamento e valores pagos por modalidade de pagamento.
print("Processando: silver_order_payments...")

df_payments = spark.table("bronze_olist.bronze_order_payments")

df_payments_silver = df_payments.select(
    col("order_id").alias("order_id"),
    col("payment_sequential").cast("integer").alias("payment_sequential"),
    col("payment_type").alias("payment_type"),
    col("payment_installments").cast("integer").alias("payment_installments"),
    col("payment_value").cast("double").alias("payment_value"),
    
    current_timestamp().alias("_processed_at")
)

df_payments_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_olist.silver_order_payments")

# 7. TABELA: SILVER_PRODUCTS (Produtos)
# Objetivo: Tipar medidas, tratar valores nulos.
print("Processando: silver_products...")

df_products = spark.table("bronze_olist.bronze_products")

df_products_silver = df_products.select(
    col("product_id").alias("product_id"),
    coalesce(trim(lower(col("product_category_name"))), lit("nao_definido")).alias("product_category_name"),
    col("product_name_lenght").cast("integer").alias("product_name_length"),
    col("product_description_lenght").cast("integer").alias("product_description_length"),
    col("product_photos_qty").cast("integer").alias("product_photos_qty"),
    col("product_weight_g").cast("double").alias("product_weight_g"),
    col("product_length_cm").cast("double").alias("product_length_cm"),
    col("product_height_cm").cast("double").alias("product_height_cm"),
    col("product_width_cm").cast("double").alias("product_width_cm"),
        
    current_timestamp().alias("_processed_at")
    )

df_products_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_olist.silver_products")

print("\n--- PROCESSAMENTO DA CAMADA SILVER FINALIZADO COM SUCESSO! ---")